In [1]:
import os
import re
import glob
import pickle
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from AnalasysFunction import plot_delta_corr_vs_mean_fr,plot_corr_two_smoothing_connected,plot_delta_corr_steps_connected_open_folder,plot_corr_multi_smoothing_connected,plot_corr_multi_smoothing_connected_open_folder

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.utils.multiclass import unique_labels
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import PrecisionRecallDisplay
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
_PAT = re.compile(r"corrDict_frSm(?P<fr>\d+(?:\.\d+)?)_calSm(?P<cal>\d+(?:\.\d+)?)\.pkl")

In [2]:
def save_plotly_svg_html(fig, svg_path, html_path):
    """
    Saves Plotly as SVG + HTML.
    Requires: pip install -U kaleido
    """
    fig.write_image(svg_path, format="svg")
    fig.write_html(html_path, include_plotlyjs="cdn")

def load_corrdict_records(folder):
    files = glob.glob(os.path.join(folder, "corrDict_frSm*_calSm*.pkl"))
    records = []
    for p in files:
        m = _PAT.search(os.path.basename(p))
        if not m:
            continue
        fr_sig = float(m.group("fr"))
        cal_sig = float(m.group("cal"))
        with open(p, "rb") as f:
            d = pickle.load(f)
        r = d.get("pearson_r", np.nan)
        try:
            r = float(r)
        except Exception:
            r = np.nan
        records.append({"fr_sig": fr_sig, "cal_sig": cal_sig, "pearson_r": r, "path": p})
    return records

def build_sigma_grid(records):
    fr_vals = np.array(sorted(set(r["fr_sig"] for r in records)), float)
    cal_vals = np.array(sorted(set(r["cal_sig"] for r in records)), float)
    r_grid = np.full((cal_vals.size, fr_vals.size), np.nan, float)
    for r in records:
        i = np.where(cal_vals == r["cal_sig"])[0][0]
        j = np.where(fr_vals == r["fr_sig"])[0][0]
        r_grid[i, j] = r["pearson_r"]
    return fr_vals, cal_vals, r_grid

def find_global_optimum(fr_vals, cal_vals, r_grid, use_abs=False):
    """Returns (best_fr, best_cal, best_r)."""
    score = np.abs(r_grid) if use_abs else r_grid
    if not np.isfinite(score).any():
        return np.nan, np.nan, np.nan
    i, j = np.unravel_index(np.nanargmax(score), score.shape)
    return float(fr_vals[j]), float(cal_vals[i]), float(r_grid[i, j])

def plot_opt_corr_plotly(folder, *, cal_sig_fixed, fr_sig_fixed, use_abs=False):
    records = load_corrdict_records(folder)
    if len(records) == 0:
        raise ValueError(f"No corrDict_frSm*_calSm*.pkl found in {folder}")

    fr_vals, cal_vals, r_grid = build_sigma_grid(records)

    # curves
    fr_curve = None
    cal_curve = None
    if cal_sig_fixed in cal_vals:
        i = np.where(cal_vals == cal_sig_fixed)[0][0]
        fr_curve = r_grid[i, :]
    if fr_sig_fixed in fr_vals:
        j = np.where(fr_vals == fr_sig_fixed)[0][0]
        cal_curve = r_grid[:, j]

    def score(x):
        return np.abs(x) if use_abs else x

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"r vs FR sigma (cal={cal_sig_fixed})",
            f"r vs Ca sigma (fr={fr_sig_fixed})"
        )
    )

    # left
    if fr_curve is not None and fr_vals.size > 0:
        fig.add_trace(go.Scatter(x=fr_vals, y=fr_curve, mode="lines+markers"), row=1, col=1)
        idx = int(np.nanargmax(score(fr_curve)))
        fig.add_trace(go.Scatter(
            x=[fr_vals[idx]], y=[fr_curve[idx]],
            mode="markers", marker=dict(size=14, symbol="star"),
            name="best"
        ), row=1, col=1)

    # right
    if cal_curve is not None and cal_vals.size > 0:
        fig.add_trace(go.Scatter(x=cal_vals, y=cal_curve, mode="lines+markers"), row=1, col=2)
        idx = int(np.nanargmax(score(cal_curve)))
        fig.add_trace(go.Scatter(
            x=[cal_vals[idx]], y=[cal_curve[idx]],
            mode="markers", marker=dict(size=14, symbol="star"),
            name="best"
        ), row=1, col=2)

    fig.update_layout(template="plotly_white", height=420, width=1050, showlegend=False)

    best_fr, best_cal, best_r = find_global_optimum(fr_vals, cal_vals, r_grid, use_abs=use_abs)

    resultD = {
        "records_list": records,
        "fr_vals": fr_vals,
        "cal_vals": cal_vals,
        "r_grid": r_grid,
        "best_fr_sig": best_fr,
        "best_cal_sig": best_cal,
        "best_r": best_r,
    }
    return fig, resultD

def _nearest_value(arr, target):
    """Return (value, index) in arr closest to target."""
    arr = np.asarray(arr, float)
    idx = int(np.nanargmin(np.abs(arr - float(target))))
    return float(arr[idx]), idx

def plot_opt_corr_plotly_three_traces(folder, *, use_abs=False):
    records = load_corrdict_records(folder)
    if len(records) == 0:
        raise ValueError(f"No corrDict_frSm*_calSm*.pkl found in {folder}")

    fr_vals, cal_vals, r_grid = build_sigma_grid(records)

    def score(x):
        return np.abs(x) if use_abs else x

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "r vs FR sigma",
            "r vs Ca sigma"
        )
    )

    # Fixed sigmas for traces
    fixed_sigs = [0.01, 0.45]

    # Left subplot: r vs FR sigma
    for cal_fixed in fixed_sigs:
        color = 'red' if cal_fixed == 0.01 else 'green'
        if cal_fixed in cal_vals:
            i = np.where(cal_vals == cal_fixed)[0][0]
            fr_curve = r_grid[i, :]
            fig.add_trace(go.Scatter(x=fr_vals, y=fr_curve, mode="lines+markers", line=dict(color=color), marker=dict(color=color), name=f"cal={cal_fixed}"), row=1, col=1)
            idx = int(np.nanargmax(score(fr_curve)))
            fig.add_trace(go.Scatter(
                x=[fr_vals[idx]], y=[fr_curve[idx]],
                mode="markers", marker=dict(size=14, symbol="star", color=color),
                name=f"best cal={cal_fixed}"
            ), row=1, col=1)

    # Diagonal for left: where fr_sig == cal_sig
    diag_fr = []
    diag_r_left = []
    for j, fr in enumerate(fr_vals):
        cal_closest, i = _nearest_value(cal_vals, fr)
        if abs(cal_closest - fr) < 1e-6:  # exact match
            diag_fr.append(fr)
            diag_r_left.append(r_grid[i, j])
    if diag_fr:
        color = 'blue'
        fig.add_trace(go.Scatter(x=diag_fr, y=diag_r_left, mode="lines+markers", line=dict(color=color), marker=dict(color=color), name="diagonal"), row=1, col=1)
        idx = int(np.nanargmax(score(np.array(diag_r_left))))
        fig.add_trace(go.Scatter(
            x=[diag_fr[idx]], y=[diag_r_left[idx]],
            mode="markers", marker=dict(size=14, symbol="star", color=color),
            name="best diagonal"
        ), row=1, col=1)

    # Right subplot: r vs Ca sigma
    for fr_fixed in fixed_sigs:
        color = 'red' if fr_fixed == 0.01 else 'green'
        if fr_fixed in fr_vals:
            j = np.where(fr_vals == fr_fixed)[0][0]
            cal_curve = r_grid[:, j]
            fig.add_trace(go.Scatter(x=cal_vals, y=cal_curve, mode="lines+markers", line=dict(color=color), marker=dict(color=color), name=f"fr={fr_fixed}"), row=1, col=2)
            idx = int(np.nanargmax(score(cal_curve)))
            fig.add_trace(go.Scatter(
                x=[cal_vals[idx]], y=[cal_curve[idx]],
                mode="markers", marker=dict(size=14, symbol="star", color=color),
                name=f"best fr={fr_fixed}"
            ), row=1, col=2)

    # Diagonal for right: where cal_sig == fr_sig
    diag_cal = []
    diag_r_right = []
    for i, cal in enumerate(cal_vals):
        fr_closest, j = _nearest_value(fr_vals, cal)
        if abs(fr_closest - cal) < 1e-6:  # exact match
            diag_cal.append(cal)
            diag_r_right.append(r_grid[i, j])
    if diag_cal:
        color = 'blue'
        fig.add_trace(go.Scatter(x=diag_cal, y=diag_r_right, mode="lines+markers", line=dict(color=color), marker=dict(color=color), name="diagonal"), row=1, col=2)
        idx = int(np.nanargmax(score(np.array(diag_r_right))))
        fig.add_trace(go.Scatter(
            x=[diag_cal[idx]], y=[diag_r_right[idx]],
            mode="markers", marker=dict(size=14, symbol="star", color=color),
            name="best diagonal"
        ), row=1, col=2)

    fig.update_layout(template="plotly_white", height=420, width=1050, showlegend=True)

    best_fr, best_cal, best_r = find_global_optimum(fr_vals, cal_vals, r_grid, use_abs=use_abs)

    resultD = {
        "records_list": records,
        "fr_vals": fr_vals,
        "cal_vals": cal_vals,
        "r_grid": r_grid,
        "best_fr_sig": best_fr,
        "best_cal_sig": best_cal,
        "best_r": best_r,
    }
    return fig, resultD

In [3]:

def _nearest_value(arr, target):
    """Return (value, index) in arr closest to target."""
    arr = np.asarray(arr, float)
    idx = int(np.nanargmin(np.abs(arr - float(target))))
    return float(arr[idx]), idx

def best_fr_at_fixed_cal(fr_vals, cal_vals, r_grid, cal_sig_fixed, *, use_abs=False):
    """Best fr (and r) along row cal_sig_fixed. Uses nearest cal if not exact."""
    if cal_vals.size == 0 or fr_vals.size == 0:
        return np.nan, np.nan, np.nan
    cal_used, i = _nearest_value(cal_vals, cal_sig_fixed)
    row = r_grid[i, :]
    score = np.abs(row) if use_abs else row
    if not np.isfinite(score).any():
        return cal_used, np.nan, np.nan
    j = int(np.nanargmax(score))
    return cal_used, float(fr_vals[j]), float(row[j])

def best_cal_at_fixed_fr(fr_vals, cal_vals, r_grid, fr_sig_fixed, *, use_abs=False):
    """Best cal (and r) along column fr_sig_fixed. Uses nearest fr if not exact."""
    if cal_vals.size == 0 or fr_vals.size == 0:
        return np.nan, np.nan, np.nan
    fr_used, j = _nearest_value(fr_vals, fr_sig_fixed)
    col = r_grid[:, j]
    score = np.abs(col) if use_abs else col
    if not np.isfinite(score).any():
        return fr_used, np.nan, np.nan
    i = int(np.nanargmax(score))
    return fr_used, float(cal_vals[i]), float(col[i])

def best_global_pair(fr_vals, cal_vals, r_grid, *, use_abs=False):
    score = np.abs(r_grid) if use_abs else r_grid
    if not np.isfinite(score).any():
        return np.nan, np.nan, np.nan
    i, j = np.unravel_index(np.nanargmax(score), score.shape)
    return float(fr_vals[j]), float(cal_vals[i]), float(r_grid[i, j])

In [4]:
def compute_all_optima_for_cell(folder, *, cal_sig_fixed, fr_sig_fixed, use_abs=False):
    records = load_corrdict_records(folder)
    if len(records) == 0:
        return None

    fr_vals, cal_vals, r_grid = build_sigma_grid(records)

    # (1) Global best pair
    g_fr, g_cal, g_r = best_global_pair(fr_vals, cal_vals, r_grid, use_abs=use_abs)

    # (2) Best FR given fixed Ca (nearest match used)
    cal_used, best_fr, best_fr_r = best_fr_at_fixed_cal(
        fr_vals, cal_vals, r_grid, cal_sig_fixed, use_abs=use_abs
    )

    # (3) Best Ca given fixed FR (nearest match used)
    fr_used, best_cal, best_cal_r = best_cal_at_fixed_fr(
        fr_vals, cal_vals, r_grid, fr_sig_fixed, use_abs=use_abs
    )

    return {
        "folder": folder,
        # global
        "best_global_fr": g_fr,
        "best_global_cal": g_cal,
        "best_global_r": g_r,

        # conditional optima
        "cal_fixed_requested": float(cal_sig_fixed),
        "cal_fixed_used": cal_used,
        "best_fr_at_cal_fixed": best_fr,
        "best_r_at_cal_fixed": best_fr_r,

        "fr_fixed_requested": float(fr_sig_fixed),
        "fr_fixed_used": fr_used,
        "best_cal_at_fr_fixed": best_cal,
        "best_r_at_fr_fixed": best_cal_r,

        # for debugging / optional plotting later
        "fr_vals": fr_vals,
        "cal_vals": cal_vals,
        "r_grid": r_grid,
    }

In [5]:
DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\SST_Final.csv')
values = DB['SNR'].tolist()
r = DB
awakePyr = r['Notes']
bsPyr = list(r['brainState'])
pathPyr = list(r['Link'])
AllCalSR = list(r['CALsr'])

In [6]:
fr_sig = 0.15
cal_sig = 0.15
corr_summary_rows = []

for i, folder in enumerate(pathPyr):
    out = compute_all_optima_for_cell(
        folder,
        cal_sig_fixed=cal_sig,  # your chosen fixed Ca sigma
        fr_sig_fixed=fr_sig,    # your chosen fixed FR sigma
        use_abs=False
    )
    fig, x = plot_opt_corr_plotly_three_traces(folder, use_abs=False)
    if out is None:
        continue
    path_html = os.path.join(folder, f"smothh_effect{fr_sig}.html")
    # Optional: save
    fig.write_html(path_html, include_plotlyjs="cdn")
    fig.write_image(os.path.join(folder, f"smothh_effect{fr_sig}.svg"))   
    corr_summary_rows.append({
        "cell_idx": i,
        "folder": folder,

        "best_global_fr": out["best_global_fr"],
        "best_global_cal": out["best_global_cal"],
        "best_global_r": out["best_global_r"],

        "cal_fixed_requested": out["cal_fixed_requested"],
        "cal_fixed_used": out["cal_fixed_used"],
        "best_fr_at_cal_fixed": out["best_fr_at_cal_fixed"],
        "best_r_at_cal_fixed": out["best_r_at_cal_fixed"],

        "fr_fixed_requested": out["fr_fixed_requested"],
        "fr_fixed_used": out["fr_fixed_used"],
        "best_cal_at_fr_fixed": out["best_cal_at_fr_fixed"],
        "best_r_at_fr_fixed": out["best_r_at_fr_fixed"],
    })

c:\Users\owner\AppData\Local\Programs\Python\Python310\lib\site-packages\kaleido\scopes\base.py:185: DeprecationWarning:

setDaemon() is deprecated, set the daemon attribute instead



In [7]:
m_path = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026\SST\paramsEX"
path_html1 = os.path.join(m_path, "Corr_change_for_meanFR_MAXS_1.5_minS_0.01.html")
path_svg1 = os.path.join(m_path, "Corr_change_for_meanFR_MAXS_1.5_minS_0.01.svg")
fig, df = plot_delta_corr_vs_mean_fr(pathPyr, fr_sig_a=0.15, cal_sig_a=0.01, fr_sig_b=1.5, cal_sig_b=1.5,delta_order="b_minus_a",save_html=path_html1,save_svg=path_svg1)
# if you want the opposite sign:
# fig, df = plot_delta_corr_vs_mean_fr(AllPath, delta_order="b_minus_a")
path_html2 = os.path.join(m_path, "Corr_cellToCell_MAXS_1.5_minS_0.15.html")
path_svg2 = os.path.join(m_path, "Corr_cellToCell_MAXS_1.5_minS_0.15.svg")
fig, df = plot_corr_two_smoothing_connected(
    pathPyr,  # list of cell folders that contain the corrDict*.pkl files
    fr_sig_a=0.15, cal_sig_a=0.15,
    fr_sig_b=1.5,  cal_sig_b=1.5,save_html=path_html2,save_svg=path_svg2
)
path_html3 = os.path.join(m_path, "Corr_cellToCell_smoot_3_1.5_0.45_0.15_0.01.html")
path_svg3 = os.path.join(m_path, "Corr_cellToCell_smoot_3_1.5_0.45_0.15_0.01.svg")
fig, df = plot_corr_multi_smoothing_connected_open_folder(
    pathPyr,
    sigmas=[0.01, 0.15,0.45, 1.5,3],
    save_html=path_html3,save_svg=path_svg3,
    show=True,
)
path_html4 = os.path.join(m_path, "Corr_change_cellToCell_smoot_3_1.5_0.45_0.15_0.01.html")
path_svg4 = os.path.join(m_path, "Corr_change_cellToCell_smoot_3_1.5_0.45_0.15_0.01.svg")

fig, df = plot_delta_corr_steps_connected_open_folder(
    pathPyr,
    sig_pairs=[(0.01,0.01), (0.15,0.15), (0.45,0.45), (1.5,1.5), (3,3)],
    save_html=path_html4,save_svg=path_svg4,
    show=True,
)





FigureWidget({
    'data': [{'hovertemplate': 'cell=%{text}<br>r=%{y:.3g}<extra></extra>',
              'marker': {'color': '#1f77b4', 'size': 8},
              'mode': 'markers',
              'name': '(0.01,0.01)',
              'text': array(['cell1', 'cell0', 'cell1', 'cell0', 'cell0', 'cell0', 'cell1', 'cell0',
                             'cell0', 'cell1', 'cell1', 'cell1', 'cell0', 'cell0', 'cell1', 'cell2',
                             'cell0', 'cell1', 'cell2', 'cell0', 'cell0', 'cell0', 'cell1', 'cell0',
                             'cell0', 'cell0', 'cell0', 'cell1', 'cell0', 'cell1', 'cell0', 'cell1',
                             'cell0', 'cell0', 'cell1', 'cell0', 'cell0', 'cell2', 'cell0', 'cell0',
                             'cell0', 'cell0', 'cell0', 'cell1', 'cell0', 'cell0', 'cell1', 'cell1',
                             'cell0', 'cell1', 'cell0', 'cell0', 'cell0', 'cell1', 'cell0', 'cell1',
                             'cell0', 'cell0', 'cell0', 'cell0', 'cell0', '

FigureWidget({
    'data': [{'hoverinfo': 'skip',
              'line': {'color': 'rgba(120,120,120,0.35)', 'width': 1},
              'mode': 'lines',
              'showlegend': False,
              'type': 'scatter',
              'uid': '23c0d8d7-350f-472b-b634-d9e461d77020',
              'x': [0.0, 1.0, 2.0, ..., 2.0, 3.0, None],
              'y': [0.044888923870737826, 0.061897108989487915,
                    0.11459010080514181, ..., 0.11459010080514181,
                    0.029496062472088247, None]},
             {'customdata': array([['Z:\\Adam-Lab-Shared\\Data\\Michal_Rubin\\srugc17\\Xb\\17-06-2025\\fov1\\cell1',
                                    'file:///Z:/Adam-Lab-Shared/Data/Michal_Rubin/srugc17/Xb/17-06-2025/fov1/cell1'],
                                   ['Z:\\Adam-Lab-Shared\\Data\\Michal_Rubin\\srugc17\\Xb\\14-07-2025\\fov7\\cell0',
                                    'file:///Z:/Adam-Lab-Shared/Data/Michal_Rubin/srugc17/Xb/14-07-2025/fov7/cell0'],
           

In [8]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.DataFrame(corr_summary_rows).dropna(subset=["best_fr_at_cal_fixed"])
# ----------------------------
# 2 boxplots in ONE figure (2 subplots)
# left: best FR at fixed Ca
# right: best Ca at fixed FR
# ----------------------------
def plot_conditional_optima_boxplots(df, *, cal_sig_fixed, fr_sig_fixed, title=None):
    """
    df must contain columns:
      - best_fr_at_cal_fixed
      - best_cal_at_fr_fixed
    (optionally) cell_idx, folder, best_r_at_cal_fixed, best_r_at_fr_fixed

    Returns Plotly fig.
    """
    d = df.copy()

    # Keep finite values
    d_fr = d[np.isfinite(d["best_fr_at_cal_fixed"].astype(float))].copy()
    d_cal = d[np.isfinite(d["best_cal_at_fr_fixed"].astype(float))].copy()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"Best FR sigma (given Ca sigma = {cal_sig_fixed})",
            f"Best Ca sigma (given FR sigma = {fr_sig_fixed})",
        )
    )

    # Left box: FR
    fig.add_trace(
        go.Box(
            y=d_fr["best_fr_at_cal_fixed"],
            name="best_fr_at_cal_fixed",
            boxpoints="all",   # show all points
            jitter=0.35,
            pointpos=0,
        ),
        row=1, col=1
    )

    # Right box: Ca
    fig.add_trace(
        go.Box(
            y=d_cal["best_cal_at_fr_fixed"],
            name="best_cal_at_fr_fixed",
            boxpoints="all",
            jitter=0.35,
            pointpos=0,
        ),
        row=1, col=2
    )

    fig.update_yaxes(title_text="FR sigma (s)", row=1, col=1)
    fig.update_yaxes(title_text="Ca sigma (s)", row=1, col=2)

    if title is None:
        title = "Conditional optimal smoothing across cells"
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=450,
        width=1000,
        showlegend=False,
    )

    fig.show()
    return fig


# ----------------------------
# Example usage after your loop
# ----------------------------
# df = pd.DataFrame(corr_summary_rows)
# fig = plot_conditional_optima_boxplots(df, cal_sig_fixed=cal_sig, fr_sig_fixed=fr_sig)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ----------------------------
# 2 boxplots in ONE figure (2 subplots)
# left: best FR at fixed Ca
# right: best Ca at fixed FR
# ----------------------------
def plot_conditional_optima_boxplots(df, *, cal_sig_fixed, fr_sig_fixed, title=None):
    """
    df must contain columns:
      - best_fr_at_cal_fixed
      - best_cal_at_fr_fixed
    (optionally) cell_idx, folder, best_r_at_cal_fixed, best_r_at_fr_fixed

    Returns Plotly fig.
    """
    d = df.copy()

    # Keep finite values
    d_fr = d[np.isfinite(d["best_fr_at_cal_fixed"].astype(float))].copy()
    d_cal = d[np.isfinite(d["best_cal_at_fr_fixed"].astype(float))].copy()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"Best FR sigma (given Ca sigma = {cal_sig_fixed})",
            f"Best Ca sigma (given FR sigma = {fr_sig_fixed})",
        )
    )

    # Left box: FR
    fig.add_trace(
        go.Box(
            y=d_fr["best_fr_at_cal_fixed"],
            name="best_fr_at_cal_fixed",
            boxpoints="all",   # show all points
            jitter=0.35,
            pointpos=0,
        ),
        row=1, col=1
    )

    # Right box: Ca
    fig.add_trace(
        go.Box(
            y=d_cal["best_cal_at_fr_fixed"],
            name="best_cal_at_fr_fixed",
            boxpoints="all",
            jitter=0.35,
            pointpos=0,
        ),
        row=1, col=2
    )

    fig.update_yaxes(title_text="FR sigma (s)", row=1, col=1)
    fig.update_yaxes(title_text="Ca sigma (s)", row=1, col=2)

    if title is None:
        title = "Conditional optimal smoothing across cells"
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=450,
        width=1000,
        showlegend=False,
    )

    fig.show()
    return fig


# ----------------------------
# Example usage after your loop
# ----------------------------
# df = pd.DataFrame(corr_summary_rows)
fig = plot_conditional_optima_boxplots(df, cal_sig_fixed=cal_sig, fr_sig_fixed=fr_sig)
mother_path = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
path_html = os.path.join(mother_path, "conditional_optima_boxplots.html")
# Optional: save
fig.write_html(path_html, include_plotlyjs="cdn")
fig.write_image(os.path.join(mother_path, "conditional_optima_boxplots.svg"))   

In [9]:
print("Total rows in df:", len(df))
print("Finite global optima:",
      np.sum(np.isfinite(df["best_global_fr"]) &
             np.isfinite(df["best_global_cal"])))

unique_pairs = df[["best_global_cal", "best_global_fr"]].drop_duplicates().shape[0]
print("Unique (cal, fr) pairs:", unique_pairs, "out of", len(df))

Total rows in df: 82
Finite global optima: 82
Unique (cal, fr) pairs: 14 out of 82


In [10]:
import numpy as np
import plotly.graph_objects as go

def scatter_pairs_with_jitter(df, jitter_x=0.005, jitter_y=0.02):
    d = df.copy()
    n = len(d)

    x = d["best_global_cal"].to_numpy(float) + np.random.normal(0, jitter_x, n)
    y = d["best_global_fr"].to_numpy(float) + np.random.normal(0, jitter_y, n)
    r = d["best_global_r"].to_numpy(float)

    text = d["cell_idx"].to_numpy() if "cell_idx" in d.columns else np.arange(n)

    fig = go.Figure(go.Scatter(
        x=x,
        y=y,
        mode="markers",
        marker=dict(
            size=9,
            opacity=0.7,
            color=r,
            colorscale="Viridis",
            showscale=True,
            colorbar=dict(title="best r"),
        ),
        text=text,
        hovertemplate="Cell: %{text}<br>Ca sigma: %{x:.3f}<br>FR sigma: %{y:.3f}<br>r: %{marker.color:.3f}<extra></extra>",
    ))

    fig.update_layout(
        template="plotly_white",
        title="Global optimal smoothing pairs (jittered to reveal overlaps)",
        xaxis_title="Optimal Ca sigma (s)",
        yaxis_title="Optimal FR sigma (s)",
        height=520,
        width=650
    )
    fig.show()
    return fig

fig_scatter = scatter_pairs_with_jitter(df)



path_html = os.path.join(mother_path, "global_optimal_pairs.html")
# Optional: save
fig_scatter.write_html(path_html, include_plotlyjs="cdn")
fig_scatter.write_image(os.path.join(mother_path, "global_optimal_pairs.svg"))

In [19]:
### Subgroup-specific analysis

import pandas as pd

# re-read metadata (DB is already available earlier, this is safe/redundant)
DB = pd.read_csv(r'Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\SST_Final.csv')

# keep only the link+subgroup columns and merge with df produced above
DB = DB[['Link','subGroup']]
df_group = df.merge(DB, left_on='folder', right_on='Link', how='left')

# normalize subgroup labels to lowercase so that 'Trans'=='trans'
df_group['subGroup'] = df_group['subGroup'].astype(str).str.lower().str.strip()
df_group['subGroup'] = df_group['subGroup'].replace({'nan':'unknown'})
print("merged rows:", len(df_group))
print("subgroups:", df_group['subGroup'].unique())

import numpy as np
import plotly.express as px

# global-optimal scatter coloured by subgroup
subs = df_group.dropna(subset=['best_global_cal','best_global_fr']).copy()
subs['x_jit'] = subs['best_global_cal'] + np.random.normal(0,0.005,len(subs))
subs['y_jit'] = subs['best_global_fr']  + np.random.normal(0,0.02,len(subs))
# use absolute r for marker size (Plotly rejects negative sizes)
subs['r_size'] = np.abs(subs['best_global_r'].fillna(0))

fig_scatter_grp = px.scatter(
    subs,
    x='x_jit', y='y_jit',
    color='subGroup',
    size='r_size',            # positive values only
    hover_data=['folder','best_global_r','subGroup'],
    title='Global optimal pairs coloured by subgroup',
    labels={'x_jit':'Optimal Ca sigma (s)', 'y_jit':'Optimal FR sigma (s)'}
)
fig_scatter_grp.update_layout(template='plotly_white')
fig_scatter_grp.show()

# optional save
# fig_scatter_grp.write_html('global_pairs_by_subgroup.html', include_plotlyjs='cdn')
# fig_scatter_grp.write_image('global_pairs_by_subgroup.svg')

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# conditional group boxplots in a single-row, two-column figure
groups = sorted(df_group['subGroup'].dropna().unique())
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=["FR sigma (Ca fixed)", "Ca sigma (FR fixed)"]
)

for g in groups:
    sub = df_group[df_group['subGroup'] == g]
    fig.add_trace(
        go.Box(
            y=sub['best_fr_at_cal_fixed'],
            name=g,
            boxpoints="all",  # show individual points
            jitter=0.35,
            pointpos=0,
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Box(
            y=sub['best_cal_at_fr_fixed'],
            name=g,
            boxpoints="all",  # show individual points
            jitter=0.35,
            pointpos=0,
        ),
        row=1, col=2
    )

fig.update_layout(
    template='plotly_white',
    height=500,
    width=1000,
    title="Conditional optimal smoothing by subgroup"
)
fig.update_yaxes(title_text='FR sigma (s)', row=1, col=1)
fig.update_yaxes(title_text='Ca sigma (s)', row=1, col=2)
fig.show()

path_html = os.path.join(mother_path, "conditional_optima_by_subgroup.html")
path_svg = os.path.join(mother_path, "conditional_optima_by_subgroup.svg")

# optional save
fig.write_html(path_html, include_plotlyjs='cdn')
fig.write_image(path_svg)

merged rows: 84
subgroups: ['trans' 'combine' 'slow' 'motion' 'negcor' 'lowcor']


In [20]:

# ----- new scatter of correlation change vs optimal sigma -----
# first compute base correlation (sig=0 for both fr and cal) for each cell
base_rs = []
for folder in df_group['folder']:
    recs = load_corrdict_records(folder)
    base = np.nan
    for r in recs:
        if r['fr_sig'] == 0.01 and r['cal_sig'] == 0.01:
            base = r['pearson_r']
            break
    base_rs.append(base)

subs2 = df_group.dropna(subset=['best_global_cal','best_global_fr','best_global_r']).copy()
subs2['base_r'] = base_rs
# compute delta relative to base correlation
subs2['delta_r_cal'] = subs2['best_r_at_cal_fixed'] - subs2['base_r']
subs2['delta_r_fr'] = subs2['best_r_at_fr_fixed'] - subs2['base_r']
# also global improvement
# delta between global optimum and base (no smoothing)
subs2['delta_r_global'] = subs2['best_global_r'] - subs2['base_r']
# jitter for visibility
subs2['cal_jit'] = subs2['best_global_cal'] + np.random.normal(0,0.005,len(subs2))
subs2['fr_jit'] = subs2['best_global_fr'] + np.random.normal(0,0.02,len(subs2))

# scatter vs Ca sigma
# determine consistent color mapping for subgroups
groups = sorted(subs2['subGroup'].dropna().unique())
color_seq = px.colors.qualitative.Plotly
color_map = {g: color_seq[i % len(color_seq)] for i, g in enumerate(groups)}

fig_delta_cal = px.scatter(
    subs2, x='cal_jit', y='delta_r_cal',
    color='subGroup',
    color_discrete_map=color_map,
    title='Δr relative to base (sig=0) vs optimal Ca sigma',
    labels={'cal_jit':'Optimal Ca sigma (s)', 'delta_r_cal':'Δr (Ca-fixed - base)'},
    opacity=0.8
)
fig_delta_cal.update_traces(marker={'size':8})  # show individual points
fig_delta_cal.update_layout(template='plotly_white')
fig_delta_cal.show()

# scatter vs FR sigma (conditional improvement)
fig_delta_fr = px.scatter(
    subs2, x='fr_jit', y='delta_r_fr',
    color='subGroup',
    color_discrete_map=color_map,
    title='Δr (FR-fixed - base) vs optimal FR sigma',
    labels={'fr_jit':'Optimal FR sigma (s)', 'delta_r_fr':'Δr (FR-fixed - base)'},
    opacity=0.8
)
fig_delta_fr.update_traces(marker={'size':8})
fig_delta_fr.update_layout(template='plotly_white')
fig_delta_fr.show()

# optional save
# scatter vs Ca sigma (global improvement)
fig_delta_cal_glob = px.scatter(
    subs2, x='cal_jit', y='delta_r_global',
    color='subGroup',
    color_discrete_map=color_map,
    title='Δr (global - base) vs optimal Ca sigma',
    labels={'cal_jit':'Optimal Ca sigma (s)', 'delta_r_global':'Δr (global - base)'},
    opacity=0.8
)
fig_delta_cal_glob.update_traces(marker={'size':8})
fig_delta_cal_glob.update_layout(template='plotly_white')
fig_delta_cal_glob.show()

# scatter vs FR sigma (global improvement)
fig_delta_fr_glob = px.scatter(
    subs2, x='fr_jit', y='delta_r_global',
    color='subGroup',
    color_discrete_map=color_map,
    title='Δr (global - base) vs optimal FR sigma',
    labels={'fr_jit':'Optimal FR sigma (s)', 'delta_r_global':'Δr (global - base)'},
    opacity=0.8
)
fig_delta_fr_glob.update_traces(marker={'size':8})
fig_delta_fr_glob.update_layout(template='plotly_white')
fig_delta_fr_glob.show()

# scatter of raw base and global r
fig_base_global = make_subplots(rows=1, cols=2,
                                subplot_titles=['Base r (no smoothing)', 'Global optimal r'])
fig_base_global.add_trace(
    go.Scatter(x=subs2['cell_idx'], y=subs2['base_r'], mode='markers',
               marker=dict(color=[color_map[g] for g in subs2['subGroup']], size=8),
               name='base r'),
    row=1, col=1
)
fig_base_global.add_trace(
    go.Scatter(x=subs2['cell_idx'], y=subs2['best_global_r'], mode='markers',
               marker=dict(color=[color_map[g] for g in subs2['subGroup']], size=8),
               name='global r'),
    row=1, col=2
)
fig_base_global.update_layout(template='plotly_white',
                              title='Comparison of base vs global correlation',
                              height=450, width=900)
fig_base_global.show()
path_html = os.path.join(mother_path, 'corr_by_subgroup.html')
path_svg = os.path.join(mother_path, 'corr_by_subgroup.svg')
fig_base_global.write_html(path_html, include_plotlyjs='cdn')
fig_base_global.write_image(path_svg)


# optional save
# fig_delta_cal.write_html('delta_r_vs_cal.html', include_plotlyjs='cdn')
# fig_delta_fr.write_html('delta_r_vs_fr.html', include_plotlyjs='cdn')

# boxplot of delta r by subgroup (Ca vs FR) with points and consistent colors
groups = sorted(subs2['subGroup'].dropna().unique())
# reuse color_map defined earlier for scatter
fig_delta_grp = make_subplots(rows=1, cols=2,
                              subplot_titles=['Δr (Ca-fixed - base)', 'Δr (FR-fixed - base)'])
for g in groups:
    sub = subs2[subs2['subGroup'] == g]
    fig_delta_grp.add_trace(go.Box(
        y=sub['delta_r_cal'],
        name=g,
        boxpoints='all',
        jitter=0.35,
        pointpos=0,
        marker_color=color_map.get(g)
    ), row=1, col=1)
    fig_delta_grp.add_trace(go.Box(
        y=sub['delta_r_fr'],
        name=g,
        boxpoints='all',
        jitter=0.35,
        pointpos=0,
        marker_color=color_map.get(g)
    ), row=1, col=2)
fig_delta_grp.update_layout(template='plotly_white',
                             title='Delta correlation by subgroup',
                             height=450, width=900,
                             showlegend=True)
fig_delta_grp.update_yaxes(title_text='Δr', row=1, col=1)
fig_delta_grp.update_yaxes(title_text='Δr', row=1, col=2)
fig_delta_grp.show()

# save boxplot to mother_path
path_html = os.path.join(mother_path, 'delta_corr_by_subgroup.html')
path_svg = os.path.join(mother_path, 'delta_corr_by_subgroup.svg')
fig_delta_grp.write_html(path_html, include_plotlyjs='cdn')



# ----- per-cell r vs sigma plots -----
for idx, row in df_group.iterrows():
    folder = row['folder']
    recs = load_corrdict_records(folder)
    if len(recs) == 0:
        continue
    dfr = pd.DataFrame(recs)
    fig_cell = make_subplots(rows=1, cols=2,
                              subplot_titles=['r vs Ca sigma','r vs FR sigma'])
    fig_cell.add_trace(go.Scatter(x=dfr['cal_sig'], y=dfr['pearson_r'], mode='markers'), row=1, col=1)
    fig_cell.add_trace(go.Scatter(x=dfr['fr_sig'], y=dfr['pearson_r'], mode='markers'), row=1, col=2)
    fig_cell.update_layout(template='plotly_white',
                            title=f'Corr curves for cell {idx}',
                            height=400, width=800)
    # save into the cell's folder path
    safe_name = "r_vs_sigma"
    cell_folder = row['folder']
    if pd.isna(cell_folder) or not os.path.isdir(cell_folder):
        cell_folder = mother_path  # fallback
    html_path = os.path.join(cell_folder, f"{safe_name}.html")
    svg_path = os.path.join(cell_folder, f"{safe_name}.svg")
    fig_cell.write_html(html_path, include_plotlyjs='cdn')
    fig_cell.write_image(svg_path)

In [8]:
# individual box plots by subgroup
import plotly.express as px

# FR sigma boxes per subgroup
fig_fr_grp = px.box(
    df_group,
    x='subGroup',
    y='best_fr_at_cal_fixed',
    points='all',
    title='Distribution of best FR sigma (Ca fixed) by subgroup'
)
fig_fr_grp.update_layout(template='plotly_white', xaxis_title='subgroup', yaxis_title='FR sigma (s)')
fig_fr_grp.show()

# Ca sigma boxes per subgroup
fig_ca_grp = px.box(
    df_group,
    x='subGroup',
    y='best_cal_at_fr_fixed',
    points='all',
    title='Distribution of best Ca sigma (FR fixed) by subgroup'
)
fig_ca_grp.update_layout(template='plotly_white', xaxis_title='subgroup', yaxis_title='Ca sigma (s)')
fig_ca_grp.show()

# optional saving
# fig_fr_grp.write_html('fr_by_subgroup.html', include_plotlyjs='cdn')
# fig_ca_grp.write_html('ca_by_subgroup.html', include_plotlyjs='cdn')


NameError: name 'df_group' is not defined

In [14]:
import plotly.graph_objects as go
import pandas as pd



fig = go.Figure()
fig.add_trace(go.Box(
    y=df["best_fr_at_cal_fixed"],
    name=f"Best FR | Ca={cal_sig}",
    boxpoints="all",
    jitter=0.35,
    pointpos=0,
))
fig.update_layout(template="plotly_white", title="Optimal FR sigma (given fixed Ca)", yaxis_title="fr sigma (s)")
fig.show()

In [15]:
df2 = pd.DataFrame(corr_summary_rows).dropna(subset=["best_cal_at_fr_fixed"])

fig = go.Figure()
fig.add_trace(go.Box(
    y=df2["best_cal_at_fr_fixed"],
    name=f"Best Ca | FR={fr_sig}",
    boxpoints="all",
    jitter=0.35,
    pointpos=0,
))
fig.update_layout(template="plotly_white", title="Optimal Ca sigma (given fixed FR)", yaxis_title="cal sigma (s)")
fig.show()

In [16]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ----------------------------
# 2 boxplots in ONE figure (2 subplots)
# left: best FR at fixed Ca
# right: best Ca at fixed FR
# ----------------------------
def plot_conditional_optima_boxplots(df, *, cal_sig_fixed, fr_sig_fixed, title=None):
    """
    df must contain columns:
      - best_fr_at_cal_fixed
      - best_cal_at_fr_fixed
    (optionally) cell_idx, folder, best_r_at_cal_fixed, best_r_at_fr_fixed

    Returns Plotly fig.
    """
    d = df.copy()

    # Keep finite values
    d_fr = d[np.isfinite(d["best_fr_at_cal_fixed"].astype(float))].copy()
    d_cal = d[np.isfinite(d["best_cal_at_fr_fixed"].astype(float))].copy()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"Best FR sigma (given Ca sigma = {cal_sig_fixed})",
            f"Best Ca sigma (given FR sigma = {fr_sig_fixed})",
        )
    )

    # Left box: FR
    fig.add_trace(
        go.Box(
            y=d_fr["best_fr_at_cal_fixed"],
            name="best_fr_at_cal_fixed",
            boxpoints="all",   # show all points
            jitter=0.35,
            pointpos=0,
        ),
        row=1, col=1
    )

    # Right box: Ca
    fig.add_trace(
        go.Box(
            y=d_cal["best_cal_at_fr_fixed"],
            name="best_cal_at_fr_fixed",
            boxpoints="all",
            jitter=0.35,
            pointpos=0,
        ),
        row=1, col=2
    )

    fig.update_yaxes(title_text="FR sigma (s)", row=1, col=1)
    fig.update_yaxes(title_text="Ca sigma (s)", row=1, col=2)

    if title is None:
        title = "Conditional optimal smoothing across cells"
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=450,
        width=1000,
        showlegend=False,
    )

    fig.show()
    return fig


# ----------------------------
# Example usage after your loop
# ----------------------------
# df = pd.DataFrame(corr_summary_rows)
# fig = plot_conditional_optima_boxplots(df, cal_sig_fixed=cal_sig, fr_sig_fixed=fr_sig)

# Optional: save
# fig.write_html(r"Z:\...\conditional_optima_boxplots.html", include_plotlyjs="cdn")
# fig.write_image(r"Z:\...\conditional_optima_boxplots.svg")

In [18]:
# 1. Scatter plot of optimal sigma combinations
import plotly.graph_objects as go

# Add jitter to reveal overlapping points
x_jit = df['best_global_cal'] + np.random.normal(0, 0.01, len(df))
y_jit = df['best_global_fr'] + np.random.normal(0, 0.05, len(df))

fig_scatter_opt = go.Figure(go.Scatter(
    x=x_jit,
    y=y_jit,
    mode='markers',
    marker=dict(
        size=8,
        color=np.abs(df['best_global_r']),
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title='|r|'),
    ),
    customdata=np.column_stack((df['best_global_cal'], df['best_global_fr'])),
    text=[f'Cell {i}<br>r={r:.3f}' for i, r in enumerate(df['best_global_r'])],
    hovertemplate='Ca sigma: %{customdata[0]:.3f}<br>FR sigma: %{customdata[1]:.3f}<br>%{text}<extra></extra>',
))

fig_scatter_opt.update_layout(
    template='plotly_white',
    title='Optimal Sigma Combinations Across Cells (Jittered)',
    xaxis_title='Optimal Ca Sigma (s)',
    yaxis_title='Optimal FR Sigma (s)',
    height=500,
    width=600
)
fig_scatter_opt.show()

# Optional save
path_html_scatter = os.path.join(mother_path, 'optimal_sigma_scatter.html')
path_svg_scatter = os.path.join(mother_path, 'optimal_sigma_scatter.svg')
fig_scatter_opt.write_html(path_html_scatter, include_plotlyjs='cdn')
fig_scatter_opt.write_image(path_svg_scatter)

In [15]:
# 2. Histogram of optimal correlations
fig_hist_opt = go.Figure(go.Histogram(
    x=df['best_global_r'],
    xbins=dict(start=-1, end=1, size=0.1),
    marker_color='blue',
    opacity=0.7
))

fig_hist_opt.update_layout(
    template='plotly_white',
    title='Distribution of Optimal Correlations',
    xaxis_title='Optimal Correlation (r)',
    yaxis_title='Number of Cells',
    height=400,
    width=600
)
fig_hist_opt.show()

# Optional save
path_html_hist = os.path.join(mother_path, 'optimal_corr_histogram.html')
path_svg_hist = os.path.join(mother_path, 'optimal_corr_histogram.svg')
fig_hist_opt.write_html(path_html_hist, include_plotlyjs='cdn')
fig_hist_opt.write_image(path_svg_hist)

In [ ]:
### Spearman mirror of Pearson smoothing analysis
from scipy.stats import spearmanr

def _to_float_or_nan(v):
    try:
        return float(v)
    except Exception:
        return np.nan

def _calc_spearman_from_arrays(x, y):
    x = np.asarray(x, float).ravel()
    y = np.asarray(y, float).ravel()
    n = min(x.size, y.size)
    if n < 3:
        return np.nan, np.nan
    x = x[:n]
    y = y[:n]
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan, np.nan
    xf = x[m]
    yf = y[m]
    if np.std(xf) == 0 or np.std(yf) == 0:
        return np.nan, np.nan
    res = spearmanr(xf, yf)
    try:
        rho = float(res.correlation)
        p = float(res.pvalue)
    except AttributeError:
        rho = float(res[0])
        p = float(res[1])
    return rho, p

def ensure_spearman_in_corrdict_file(path_corrdict):
    """
    Ensures spearman values exist inside corrDict pkl and writes back if missing.
    Returns the loaded dict (with spearman keys available).
    """
    with open(path_corrdict, "rb") as f:
        d = pickle.load(f)

    rho_old = _to_float_or_nan(d.get("spearman_r", np.nan))
    p_old = _to_float_or_nan(d.get("spearman_p", np.nan))
    changed = False

    if not np.isfinite(rho_old) or not np.isfinite(p_old):
        rho_new, p_new = _calc_spearman_from_arrays(d.get("fr_on_ca", []), d.get("ca_used", []))
        d["spearman_r"] = float(rho_new) if np.isfinite(rho_new) else np.nan
        d["spearman_p"] = float(p_new) if np.isfinite(p_new) else np.nan
        changed = True
    else:
        if "spearman_r" not in d:
            d["spearman_r"] = float(rho_old)
            changed = True
        if "spearman_p" not in d:
            d["spearman_p"] = float(p_old)
            changed = True

    if changed:
        with open(path_corrdict, "wb") as f:
            pickle.dump(d, f)

    return d

def load_corrdict_records_spearman(folder):
    files = glob.glob(os.path.join(folder, "corrDict_frSm*_calSm*.pkl"))
    records = []
    for p in files:
        m = _PAT.search(os.path.basename(p))
        if not m:
            continue
        fr_sig = float(m.group("fr"))
        cal_sig = float(m.group("cal"))
        d = ensure_spearman_in_corrdict_file(p)
        rho = _to_float_or_nan(d.get("spearman_r", np.nan))
        records.append({"fr_sig": fr_sig, "cal_sig": cal_sig, "spearman_r": rho, "path": p})
    return records

def build_sigma_grid_spearman(records):
    fr_vals = np.array(sorted(set(r["fr_sig"] for r in records)), float)
    cal_vals = np.array(sorted(set(r["cal_sig"] for r in records)), float)
    rho_grid = np.full((cal_vals.size, fr_vals.size), np.nan, float)
    for r in records:
        i = np.where(cal_vals == r["cal_sig"])[0][0]
        j = np.where(fr_vals == r["fr_sig"])[0][0]
        rho_grid[i, j] = r["spearman_r"]
    return fr_vals, cal_vals, rho_grid

def plot_opt_spearman_plotly_three_traces(folder, *, use_abs=False):
    records = load_corrdict_records_spearman(folder)
    if len(records) == 0:
        raise ValueError(f"No corrDict_frSm*_calSm*.pkl found in {folder}")

    fr_vals, cal_vals, rho_grid = build_sigma_grid_spearman(records)

    def score(x):
        return np.abs(x) if use_abs else x

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            "rho vs FR sigma",
            "rho vs Ca sigma"
        )
    )

    fixed_sigs = [0.01, 0.45]

    for cal_fixed in fixed_sigs:
        color = "red" if cal_fixed == 0.01 else "green"
        if cal_fixed in cal_vals:
            i = np.where(cal_vals == cal_fixed)[0][0]
            fr_curve = rho_grid[i, :]
            fig.add_trace(go.Scatter(x=fr_vals, y=fr_curve, mode="lines+markers", line=dict(color=color), marker=dict(color=color), name=f"cal={cal_fixed}"), row=1, col=1)
            if np.isfinite(fr_curve).any():
                idx = int(np.nanargmax(score(fr_curve)))
                fig.add_trace(go.Scatter(x=[fr_vals[idx]], y=[fr_curve[idx]], mode="markers", marker=dict(size=14, symbol="star", color=color), name=f"best cal={cal_fixed}"), row=1, col=1)

    diag_fr = []
    diag_rho_left = []
    for j, fr in enumerate(fr_vals):
        cal_closest, i = _nearest_value(cal_vals, fr)
        if abs(cal_closest - fr) < 1e-6:
            diag_fr.append(fr)
            diag_rho_left.append(rho_grid[i, j])
    if diag_fr:
        color = "blue"
        fig.add_trace(go.Scatter(x=diag_fr, y=diag_rho_left, mode="lines+markers", line=dict(color=color), marker=dict(color=color), name="diagonal"), row=1, col=1)
        if np.isfinite(diag_rho_left).any():
            idx = int(np.nanargmax(score(np.array(diag_rho_left, float))))
            fig.add_trace(go.Scatter(x=[diag_fr[idx]], y=[diag_rho_left[idx]], mode="markers", marker=dict(size=14, symbol="star", color=color), name="best diagonal"), row=1, col=1)

    for fr_fixed in fixed_sigs:
        color = "red" if fr_fixed == 0.01 else "green"
        if fr_fixed in fr_vals:
            j = np.where(fr_vals == fr_fixed)[0][0]
            cal_curve = rho_grid[:, j]
            fig.add_trace(go.Scatter(x=cal_vals, y=cal_curve, mode="lines+markers", line=dict(color=color), marker=dict(color=color), name=f"fr={fr_fixed}"), row=1, col=2)
            if np.isfinite(cal_curve).any():
                idx = int(np.nanargmax(score(cal_curve)))
                fig.add_trace(go.Scatter(x=[cal_vals[idx]], y=[cal_curve[idx]], mode="markers", marker=dict(size=14, symbol="star", color=color), name=f"best fr={fr_fixed}"), row=1, col=2)

    diag_cal = []
    diag_rho_right = []
    for i, cal in enumerate(cal_vals):
        fr_closest, j = _nearest_value(fr_vals, cal)
        if abs(fr_closest - cal) < 1e-6:
            diag_cal.append(cal)
            diag_rho_right.append(rho_grid[i, j])
    if diag_cal:
        color = "blue"
        fig.add_trace(go.Scatter(x=diag_cal, y=diag_rho_right, mode="lines+markers", line=dict(color=color), marker=dict(color=color), name="diagonal"), row=1, col=2)
        if np.isfinite(diag_rho_right).any():
            idx = int(np.nanargmax(score(np.array(diag_rho_right, float))))
            fig.add_trace(go.Scatter(x=[diag_cal[idx]], y=[diag_rho_right[idx]], mode="markers", marker=dict(size=14, symbol="star", color=color), name="best diagonal"), row=1, col=2)

    fig.update_layout(template="plotly_white", height=420, width=1050, showlegend=True)

    best_fr, best_cal, best_rho = best_global_pair(fr_vals, cal_vals, rho_grid, use_abs=use_abs)
    resultD = {
        "records_list": records,
        "fr_vals": fr_vals,
        "cal_vals": cal_vals,
        "rho_grid": rho_grid,
        "best_fr_sig": best_fr,
        "best_cal_sig": best_cal,
        "best_rho": best_rho,
    }
    return fig, resultD

def compute_all_optima_for_cell_spearman(folder, *, cal_sig_fixed, fr_sig_fixed, use_abs=False):
    records = load_corrdict_records_spearman(folder)
    if len(records) == 0:
        return None

    fr_vals, cal_vals, rho_grid = build_sigma_grid_spearman(records)
    g_fr, g_cal, g_rho = best_global_pair(fr_vals, cal_vals, rho_grid, use_abs=use_abs)
    cal_used, best_fr, best_fr_rho = best_fr_at_fixed_cal(fr_vals, cal_vals, rho_grid, cal_sig_fixed, use_abs=use_abs)
    fr_used, best_cal, best_cal_rho = best_cal_at_fixed_fr(fr_vals, cal_vals, rho_grid, fr_sig_fixed, use_abs=use_abs)

    return {
        "folder": folder,
        "best_global_fr": g_fr,
        "best_global_cal": g_cal,
        "best_global_rho": g_rho,
        "best_global_r": g_rho,
        "cal_fixed_requested": float(cal_sig_fixed),
        "cal_fixed_used": cal_used,
        "best_fr_at_cal_fixed": best_fr,
        "best_rho_at_cal_fixed": best_fr_rho,
        "best_r_at_cal_fixed": best_fr_rho,
        "fr_fixed_requested": float(fr_sig_fixed),
        "fr_fixed_used": fr_used,
        "best_cal_at_fr_fixed": best_cal,
        "best_rho_at_fr_fixed": best_cal_rho,
        "best_r_at_fr_fixed": best_cal_rho,
        "fr_vals": fr_vals,
        "cal_vals": cal_vals,
        "rho_grid": rho_grid,
    }


In [ ]:
### Spearman run loop (mirrors Pearson loop)
fr_sig = 0.15
cal_sig = 0.15
corr_summary_rows_spearman = []

for i, folder in enumerate(pathPyr):
    out = compute_all_optima_for_cell_spearman(
        folder,
        cal_sig_fixed=cal_sig,
        fr_sig_fixed=fr_sig,
        use_abs=False
    )
    if out is None:
        continue

    fig_spearman, _ = plot_opt_spearman_plotly_three_traces(folder, use_abs=False)
    path_html = os.path.join(folder, f"smothh_effect{fr_sig}_spearman_correlation.html")
    path_svg = os.path.join(folder, f"smothh_effect{fr_sig}_spearman_correlation.svg")
    fig_spearman.write_html(path_html, include_plotlyjs="cdn")
    fig_spearman.write_image(path_svg)

    corr_summary_rows_spearman.append({
        "cell_idx": i,
        "folder": folder,
        "best_global_fr": out["best_global_fr"],
        "best_global_cal": out["best_global_cal"],
        "best_global_rho": out["best_global_rho"],
        "best_global_r": out["best_global_r"],
        "cal_fixed_requested": out["cal_fixed_requested"],
        "cal_fixed_used": out["cal_fixed_used"],
        "best_fr_at_cal_fixed": out["best_fr_at_cal_fixed"],
        "best_rho_at_cal_fixed": out["best_rho_at_cal_fixed"],
        "best_r_at_cal_fixed": out["best_r_at_cal_fixed"],
        "fr_fixed_requested": out["fr_fixed_requested"],
        "fr_fixed_used": out["fr_fixed_used"],
        "best_cal_at_fr_fixed": out["best_cal_at_fr_fixed"],
        "best_rho_at_fr_fixed": out["best_rho_at_fr_fixed"],
        "best_r_at_fr_fixed": out["best_r_at_fr_fixed"],
    })

df_spearman = pd.DataFrame(corr_summary_rows_spearman).dropna(subset=["best_fr_at_cal_fixed"])
print("Spearman rows:", len(df_spearman))


In [ ]:
### Spearman conditional-optima figure + save
fig_spearman_box = plot_conditional_optima_boxplots(
    df_spearman,
    cal_sig_fixed=cal_sig,
    fr_sig_fixed=fr_sig,
    title="Conditional optimal smoothing across cells (Spearman)"
)

mother_path = r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\data summery\2026"
path_html = os.path.join(mother_path, "conditional_optima_boxplots_spearman_correlation.html")
path_svg = os.path.join(mother_path, "conditional_optima_boxplots_spearman_correlation.svg")
fig_spearman_box.write_html(path_html, include_plotlyjs="cdn")
fig_spearman_box.write_image(path_svg)


In [ ]:
### Spearman mirror: all summary figures
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Base dataframe for Spearman summary figures
if "df_spearman" not in globals():
    raise NameError("Run the Spearman loop cell first (df_spearman is missing).")

df_sp = df_spearman.copy()
if "best_global_r" not in df_sp.columns and "best_global_rho" in df_sp.columns:
    df_sp["best_global_r"] = df_sp["best_global_rho"]
if "best_r_at_cal_fixed" not in df_sp.columns and "best_rho_at_cal_fixed" in df_sp.columns:
    df_sp["best_r_at_cal_fixed"] = df_sp["best_rho_at_cal_fixed"]
if "best_r_at_fr_fixed" not in df_sp.columns and "best_rho_at_fr_fixed" in df_sp.columns:
    df_sp["best_r_at_fr_fixed"] = df_sp["best_rho_at_fr_fixed"]

# 1) Global optimal pairs scatter (jittered)
n = len(df_sp)
x = df_sp["best_global_cal"].to_numpy(float) + np.random.normal(0, 0.005, n)
y = df_sp["best_global_fr"].to_numpy(float) + np.random.normal(0, 0.02, n)
rho = df_sp["best_global_r"].to_numpy(float)
text = df_sp["cell_idx"].to_numpy() if "cell_idx" in df_sp.columns else np.arange(n)
fig_scatter_sp = go.Figure(go.Scatter(
    x=x, y=y, mode="markers",
    marker=dict(size=9, opacity=0.7, color=rho, colorscale="Viridis", showscale=True, colorbar=dict(title="best rho")),
    text=text,
    hovertemplate="Cell: %{text}<br>Ca sigma: %{x:.3f}<br>FR sigma: %{y:.3f}<br>rho: %{marker.color:.3f}<extra></extra>",
))
fig_scatter_sp.update_layout(template="plotly_white", title="Global optimal smoothing pairs (Spearman, jittered)", xaxis_title="Optimal Ca sigma (s)", yaxis_title="Optimal FR sigma (s)", height=520, width=650)
fig_scatter_sp.show()
fig_scatter_sp.write_html(os.path.join(mother_path, "global_optimal_pairs_spearman_correlation.html"), include_plotlyjs="cdn")
fig_scatter_sp.write_image(os.path.join(mother_path, "global_optimal_pairs_spearman_correlation.svg"))

# 2) Subgroup merge and subgroup summary figures
DB_sp = pd.read_csv(r"Z:\Adam-Lab-Shared\Data\Michal_Rubin\Dendrites\SST_Final.csv")[["Link", "subGroup"]]
df_group_sp = df_sp.merge(DB_sp, left_on="folder", right_on="Link", how="left")
df_group_sp["subGroup"] = df_group_sp["subGroup"].astype(str).str.lower().str.strip().replace({"nan": "unknown"})

subs = df_group_sp.dropna(subset=["best_global_cal", "best_global_fr"]).copy()
subs["x_jit"] = subs["best_global_cal"] + np.random.normal(0, 0.005, len(subs))
subs["y_jit"] = subs["best_global_fr"] + np.random.normal(0, 0.02, len(subs))
subs["r_size"] = np.abs(subs["best_global_r"].fillna(0))
fig_scatter_grp_sp = px.scatter(subs, x="x_jit", y="y_jit", color="subGroup", size="r_size", hover_data=["folder", "best_global_r", "subGroup"], title="Global optimal pairs by subgroup (Spearman)", labels={"x_jit": "Optimal Ca sigma (s)", "y_jit": "Optimal FR sigma (s)"})
fig_scatter_grp_sp.update_layout(template="plotly_white")
fig_scatter_grp_sp.show()
fig_scatter_grp_sp.write_html(os.path.join(mother_path, "global_pairs_by_subgroup_spearman_correlation.html"), include_plotlyjs="cdn")
fig_scatter_grp_sp.write_image(os.path.join(mother_path, "global_pairs_by_subgroup_spearman_correlation.svg"))

groups = sorted(df_group_sp["subGroup"].dropna().unique())
fig_cond_grp_sp = make_subplots(rows=1, cols=2, subplot_titles=["FR sigma (Ca fixed)", "Ca sigma (FR fixed)"])
for g in groups:
    sub = df_group_sp[df_group_sp["subGroup"] == g]
    fig_cond_grp_sp.add_trace(go.Box(y=sub["best_fr_at_cal_fixed"], name=g, boxpoints="all", jitter=0.35, pointpos=0), row=1, col=1)
    fig_cond_grp_sp.add_trace(go.Box(y=sub["best_cal_at_fr_fixed"], name=g, boxpoints="all", jitter=0.35, pointpos=0), row=1, col=2)
fig_cond_grp_sp.update_layout(template="plotly_white", height=500, width=1000, title="Conditional optimal smoothing by subgroup (Spearman)")
fig_cond_grp_sp.update_yaxes(title_text="FR sigma (s)", row=1, col=1)
fig_cond_grp_sp.update_yaxes(title_text="Ca sigma (s)", row=1, col=2)
fig_cond_grp_sp.show()
fig_cond_grp_sp.write_html(os.path.join(mother_path, "conditional_optima_by_subgroup_spearman_correlation.html"), include_plotlyjs="cdn")
fig_cond_grp_sp.write_image(os.path.join(mother_path, "conditional_optima_by_subgroup_spearman_correlation.svg"))

# 3) Individual subgroup boxplots (FR and Ca separately)
fig_fr_grp_sp = px.box(df_group_sp, x="subGroup", y="best_fr_at_cal_fixed", points="all", title="Distribution of best FR sigma (Ca fixed) by subgroup (Spearman)")
fig_fr_grp_sp.update_layout(template="plotly_white", xaxis_title="subgroup", yaxis_title="FR sigma (s)")
fig_fr_grp_sp.show()
fig_fr_grp_sp.write_html(os.path.join(mother_path, "fr_by_subgroup_spearman_correlation.html"), include_plotlyjs="cdn")
fig_fr_grp_sp.write_image(os.path.join(mother_path, "fr_by_subgroup_spearman_correlation.svg"))

fig_ca_grp_sp = px.box(df_group_sp, x="subGroup", y="best_cal_at_fr_fixed", points="all", title="Distribution of best Ca sigma (FR fixed) by subgroup (Spearman)")
fig_ca_grp_sp.update_layout(template="plotly_white", xaxis_title="subgroup", yaxis_title="Ca sigma (s)")
fig_ca_grp_sp.show()
fig_ca_grp_sp.write_html(os.path.join(mother_path, "ca_by_subgroup_spearman_correlation.html"), include_plotlyjs="cdn")
fig_ca_grp_sp.write_image(os.path.join(mother_path, "ca_by_subgroup_spearman_correlation.svg"))

# 4) Delta-vs-base Spearman figures
base_rhos = []
for folder in df_group_sp["folder"]:
    recs = load_corrdict_records_spearman(folder)
    base = np.nan
    for rr in recs:
        if rr["fr_sig"] == 0.01 and rr["cal_sig"] == 0.01:
            base = rr["spearman_r"]
            break
    base_rhos.append(base)
df_group_sp["base_r"] = base_rhos

subs2 = df_group_sp.dropna(subset=["best_global_cal", "best_global_fr", "best_global_r"]).copy()
subs2["delta_r_cal"] = subs2["best_r_at_cal_fixed"] - subs2["base_r"]
subs2["delta_r_fr"] = subs2["best_r_at_fr_fixed"] - subs2["base_r"]
subs2["delta_r_global"] = subs2["best_global_r"] - subs2["base_r"]
subs2["cal_jit"] = subs2["best_global_cal"] + np.random.normal(0, 0.005, len(subs2))
subs2["fr_jit"] = subs2["best_global_fr"] + np.random.normal(0, 0.02, len(subs2))

groups2 = sorted(subs2["subGroup"].dropna().unique())
color_seq = px.colors.qualitative.Plotly
color_map = {g: color_seq[i % len(color_seq)] for i, g in enumerate(groups2)}

fig_delta_cal_sp = px.scatter(subs2, x="cal_jit", y="delta_r_cal", color="subGroup", color_discrete_map=color_map, title="?rho (Ca-fixed - base) vs optimal Ca sigma", labels={"cal_jit":"Optimal Ca sigma (s)", "delta_r_cal":"?rho (Ca-fixed - base)"}, opacity=0.8)
fig_delta_cal_sp.update_traces(marker={"size":8})
fig_delta_cal_sp.update_layout(template="plotly_white")
fig_delta_cal_sp.show()
fig_delta_cal_sp.write_html(os.path.join(mother_path, "delta_r_vs_cal_spearman_correlation.html"), include_plotlyjs="cdn")
fig_delta_cal_sp.write_image(os.path.join(mother_path, "delta_r_vs_cal_spearman_correlation.svg"))

fig_delta_fr_sp = px.scatter(subs2, x="fr_jit", y="delta_r_fr", color="subGroup", color_discrete_map=color_map, title="?rho (FR-fixed - base) vs optimal FR sigma", labels={"fr_jit":"Optimal FR sigma (s)", "delta_r_fr":"?rho (FR-fixed - base)"}, opacity=0.8)
fig_delta_fr_sp.update_traces(marker={"size":8})
fig_delta_fr_sp.update_layout(template="plotly_white")
fig_delta_fr_sp.show()
fig_delta_fr_sp.write_html(os.path.join(mother_path, "delta_r_vs_fr_spearman_correlation.html"), include_plotlyjs="cdn")
fig_delta_fr_sp.write_image(os.path.join(mother_path, "delta_r_vs_fr_spearman_correlation.svg"))

fig_delta_cal_glob_sp = px.scatter(subs2, x="cal_jit", y="delta_r_global", color="subGroup", color_discrete_map=color_map, title="?rho (global - base) vs optimal Ca sigma", labels={"cal_jit":"Optimal Ca sigma (s)", "delta_r_global":"?rho (global - base)"}, opacity=0.8)
fig_delta_cal_glob_sp.update_traces(marker={"size":8})
fig_delta_cal_glob_sp.update_layout(template="plotly_white")
fig_delta_cal_glob_sp.show()
fig_delta_cal_glob_sp.write_html(os.path.join(mother_path, "delta_r_global_vs_cal_spearman_correlation.html"), include_plotlyjs="cdn")
fig_delta_cal_glob_sp.write_image(os.path.join(mother_path, "delta_r_global_vs_cal_spearman_correlation.svg"))

fig_delta_fr_glob_sp = px.scatter(subs2, x="fr_jit", y="delta_r_global", color="subGroup", color_discrete_map=color_map, title="?rho (global - base) vs optimal FR sigma", labels={"fr_jit":"Optimal FR sigma (s)", "delta_r_global":"?rho (global - base)"}, opacity=0.8)
fig_delta_fr_glob_sp.update_traces(marker={"size":8})
fig_delta_fr_glob_sp.update_layout(template="plotly_white")
fig_delta_fr_glob_sp.show()
fig_delta_fr_glob_sp.write_html(os.path.join(mother_path, "delta_r_global_vs_fr_spearman_correlation.html"), include_plotlyjs="cdn")
fig_delta_fr_glob_sp.write_image(os.path.join(mother_path, "delta_r_global_vs_fr_spearman_correlation.svg"))

fig_base_global_sp = make_subplots(rows=1, cols=2, subplot_titles=["Base rho (no smoothing)", "Global optimal rho"])
fig_base_global_sp.add_trace(go.Scatter(x=subs2["cell_idx"], y=subs2["base_r"], mode="markers", marker=dict(color=[color_map[g] for g in subs2["subGroup"]], size=8), name="base rho"), row=1, col=1)
fig_base_global_sp.add_trace(go.Scatter(x=subs2["cell_idx"], y=subs2["best_global_r"], mode="markers", marker=dict(color=[color_map[g] for g in subs2["subGroup"]], size=8), name="global rho"), row=1, col=2)
fig_base_global_sp.update_layout(template="plotly_white", title="Comparison of base vs global Spearman correlation", height=450, width=900)
fig_base_global_sp.show()
fig_base_global_sp.write_html(os.path.join(mother_path, "corr_by_subgroup_spearman_correlation.html"), include_plotlyjs="cdn")
fig_base_global_sp.write_image(os.path.join(mother_path, "corr_by_subgroup_spearman_correlation.svg"))

fig_delta_grp_sp = make_subplots(rows=1, cols=2, subplot_titles=["?rho (Ca-fixed - base)", "?rho (FR-fixed - base)"])
for g in groups2:
    sub = subs2[subs2["subGroup"] == g]
    fig_delta_grp_sp.add_trace(go.Box(y=sub["delta_r_cal"], name=g, boxpoints="all", jitter=0.35, pointpos=0, marker_color=color_map.get(g)), row=1, col=1)
    fig_delta_grp_sp.add_trace(go.Box(y=sub["delta_r_fr"], name=g, boxpoints="all", jitter=0.35, pointpos=0, marker_color=color_map.get(g)), row=1, col=2)
fig_delta_grp_sp.update_layout(template="plotly_white", title="Delta Spearman correlation by subgroup", height=450, width=900, showlegend=True)
fig_delta_grp_sp.update_yaxes(title_text="?rho", row=1, col=1)
fig_delta_grp_sp.update_yaxes(title_text="?rho", row=1, col=2)
fig_delta_grp_sp.show()
fig_delta_grp_sp.write_html(os.path.join(mother_path, "delta_corr_by_subgroup_spearman_correlation.html"), include_plotlyjs="cdn")
fig_delta_grp_sp.write_image(os.path.join(mother_path, "delta_corr_by_subgroup_spearman_correlation.svg"))

# 5) Per-cell rho-vs-sigma curves (saved into each cell folder)
for idx, row in df_group_sp.iterrows():
    folder = row["folder"]
    recs = load_corrdict_records_spearman(folder)
    if len(recs) == 0:
        continue
    dfr = pd.DataFrame(recs)
    fig_cell_sp = make_subplots(rows=1, cols=2, subplot_titles=["rho vs Ca sigma", "rho vs FR sigma"])
    fig_cell_sp.add_trace(go.Scatter(x=dfr["cal_sig"], y=dfr["spearman_r"], mode="markers"), row=1, col=1)
    fig_cell_sp.add_trace(go.Scatter(x=dfr["fr_sig"], y=dfr["spearman_r"], mode="markers"), row=1, col=2)
    fig_cell_sp.update_layout(template="plotly_white", title=f"Spearman curves for cell {idx}", height=400, width=800)
    cell_folder = row["folder"]
    if pd.isna(cell_folder) or (not os.path.isdir(cell_folder)):
        cell_folder = mother_path
    fig_cell_sp.write_html(os.path.join(cell_folder, "r_vs_sigma_spearman_correlation.html"), include_plotlyjs="cdn")
    fig_cell_sp.write_image(os.path.join(cell_folder, "r_vs_sigma_spearman_correlation.svg"))

# 6) Final optimal scatter + histogram mirrors
x_jit2 = df_sp["best_global_cal"] + np.random.normal(0, 0.01, len(df_sp))
y_jit2 = df_sp["best_global_fr"] + np.random.normal(0, 0.05, len(df_sp))
fig_scatter_opt_sp = go.Figure(go.Scatter(
    x=x_jit2, y=y_jit2, mode="markers",
    marker=dict(size=8, color=np.abs(df_sp["best_global_r"]), colorscale="Viridis", showscale=True, colorbar=dict(title="|rho|")),
    customdata=np.column_stack((df_sp["best_global_cal"], df_sp["best_global_fr"])),
    text=[f"Cell {i}<br>rho={r:.3f}" for i, r in enumerate(df_sp["best_global_r"])],
    hovertemplate="Ca sigma: %{customdata[0]:.3f}<br>FR sigma: %{customdata[1]:.3f}<br>%{text}<extra></extra>",
))
fig_scatter_opt_sp.update_layout(template="plotly_white", title="Optimal Sigma Combinations Across Cells (Spearman, jittered)", xaxis_title="Optimal Ca Sigma (s)", yaxis_title="Optimal FR Sigma (s)", height=500, width=600)
fig_scatter_opt_sp.show()
fig_scatter_opt_sp.write_html(os.path.join(mother_path, "optimal_sigma_scatter_spearman_correlation.html"), include_plotlyjs="cdn")
fig_scatter_opt_sp.write_image(os.path.join(mother_path, "optimal_sigma_scatter_spearman_correlation.svg"))

fig_hist_opt_sp = go.Figure(go.Histogram(x=df_sp["best_global_r"], xbins=dict(start=-1, end=1, size=0.1), marker_color="blue", opacity=0.7))
fig_hist_opt_sp.update_layout(template="plotly_white", title="Distribution of Optimal Spearman Correlations", xaxis_title="Optimal Spearman correlation (rho)", yaxis_title="Number of Cells", height=400, width=600)
fig_hist_opt_sp.show()
fig_hist_opt_sp.write_html(os.path.join(mother_path, "optimal_corr_histogram_spearman_correlation.html"), include_plotlyjs="cdn")
fig_hist_opt_sp.write_image(os.path.join(mother_path, "optimal_corr_histogram_spearman_correlation.svg"))

print("Finished Spearman summary mirrors. Files saved with suffix: _spearman_correlation")
